# Volume 3. NEMO-Style Toy Parser

Question: how does the maintained assembly-calculus parser connect word categories, roles, and word order in a controlled example?

This is a tiny grammar workshop. The sentence is intentionally plain: `dog chases cat`. The goal is to watch one word become more than a string. In this parser, a word can have a category, occupy a role, and participate in an order.

This notebook uses `assembly_calculus.NemoParser`, not the optional GPU-heavy `neural_assemblies.nemo` stack. The point is to show the package surface that currently runs in the default CPU environment.

In [ ]:
# Brain provides the shared assembly substrate for the parser.
from neural_assemblies.core.brain import Brain

# overlap lets us inspect whether role assemblies are distinct.
from neural_assemblies.assembly_calculus import overlap

# NemoParser is the maintained CPU-friendly parser surface for this toy lesson.
from neural_assemblies.assembly_calculus.parser import NemoParser

Create a small parser. The vocabulary and training set are deliberately tiny so the mechanics are visible.

In [ ]:
# Keep the parser small so the categories, roles, and order are easy to inspect.
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
parser = NemoParser(brain, n=3_000, k=50, beta=0.1, rounds=5)

# setup_areas creates the parser's internal category, role, and sequence areas.
parser.setup_areas()

# Register nouns with visual grounding labels.
for noun in ["dog", "cat"]:
    parser.register_word(noun, "noun", f"vis_{noun}")

# Register the verb with a motor grounding label.
parser.register_word("chases", "verb", "mot_chases")

Train three separate pieces: lexical category, thematic role, and word order. Keeping these stages separate makes the toy system easier to inspect and easier to falsify.

In [ ]:
# One tiny SVO sentence is enough to show the parser pipeline.
training_sentences = [["dog", "chases", "cat"]]

# Stage 1: learn word/category assemblies.
parser.train_lexicon()

# Stage 2: bind words to thematic roles such as AGENT and PATIENT.
parser.train_roles(training_sentences)

# Stage 3: memorize the observed word order.
parser.train_word_order(training_sentences)

Parse the training sentence and inspect the returned dictionary. This is a controlled smoke example, not a general language benchmark.

In [ ]:
# Parse the sentence we trained on and inspect the dictionary, not just success/failure.
result = parser.parse(["dog", "chases", "cat"])

# Categories answer: what kind of word did the parser assign?
print("categories:", result["categories"])

# Roles answer: what function did the word serve in the sentence?
print("roles:", result["roles"])

The internal role lexicons are ordinary assembly snapshots. You can inspect them with the same tools used in the foundations volume.

In [ ]:
# Role lexicons store ordinary Assembly snapshots, so we can inspect them directly.
agent_dog = parser.role_lexicons["ROLE_AGENT"]["dog"]
patient_cat = parser.role_lexicons["ROLE_PATIENT"]["cat"]

# These should live in different role areas.
print("agent area:", agent_dog.area)
print("patient area:", patient_cat.area)

# Cross-role overlap is a diagnostic, not a claim about language competence.
print("cross-role overlap:", f"{overlap(agent_dog, patient_cat):.3f}")

What to notice: this parser is compositional in the code structure. It separates category learning, role binding, and sequence memory so each part can be tested and debugged.

Try next: add `sees` as a second verb and train on `cat sees dog`. Then parse both sentences and compare the role dictionaries. Keep the vocabulary small until the mechanics are clear.